In [1]:
import os
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from pathlib import Path
import pynapple as nap

from spatial_manifolds.data.binning import get_bin_config
from spatial_manifolds.data.loading import load_session
from spatial_manifolds.detect_grids import *
from spatial_manifolds.brainrender_helper import *

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

source_path = Path('/Users/harryclark/Downloads/COHORT12/')
data_path   = '/Users/harryclark/Documents/spatial-manifolds/data'
fig_out     = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_1_registration'

mouse_days = {
    20: [14,15,16,17,18,19,20,21,22,23,24,25,26],
    21: [15,16,17,18,19,20,21,22,23,24,25,26],
    22: [33,34,35,36,37,38,39,40,41],
    25: [16,17,18,19,20,21,22,23,24,25],
    26: [11,12,13,14,15,16,17,18,19],
    27: [16,17,18,19,20,21,22,23,24,26],
    28: [16,17,18,19,20,21,22,23,25],
    29: [16,17,18,19,20,21,22,23,25],
}


In [2]:
def make_annotations(mouse, path=None, force_remake=False):
    """
    Build and save brain-area annotation arrays for the probe map.
    Set force_remake=True to delete cached files and regenerate from scratch.
    """
    Mouse = f'M{mouse}'
    data_paths = [
        f"/Users/harryclark/Documents/brainrender/probe_data/{Mouse}_probe_locations_{a}.mat"
        for a in [1, 2, 3, 4]
    ]
    shank_offsets_SC = pd.read_csv('/Users/harryclark/Documents/brainrender/probe_data/shank_offsets.csv')
    clusters_df = pd.read_csv(
        '/Users/harryclark/Documents/brainrender/probe_data/extremum_channel_locations_kilosort4_0.csv'
    )
    clusters_df      = clusters_df[clusters_df['mouse'] == mouse]
    shank_offsets_SC = shank_offsets_SC[shank_offsets_SC['mouse'] == mouse]
    clusters_df      = reconstruct_shank_id(clusters_df, mouse)

    probes_locs              = [read_probe_mat(p) for p in data_paths]
    adjusted_probes_locs_CCF = np.array([adjust_probe_locs(pl) for pl in probes_locs])
    adjusted_probe_locs_SC, adjusted_probe_locs_CCF = correct_for_left_side(adjusted_probes_locs_CCF)
    adjusted_probe_locs_SC, adjusted_probe_locs_CCF = adjust_to_shank_offsets(
        adjusted_probe_locs_SC, shank_offsets_SC
    )

    ymin, ymax = -300, 3000
    xmin, xmax = -200, 1000
    xs = np.arange(xmin, xmax, 1)
    ys = np.arange(ymin, ymax, 1)

    ann_path = f"{path}/M{mouse}_annotations.npy"
    col_path = f"{path}/M{mouse}_annotation_colors.npy"

    if force_remake:
        for p in [ann_path, col_path]:
            if os.path.exists(p):
                os.remove(p)
                print(f"  [make_annotations] Removed cache: {p}")

    if os.path.exists(ann_path):
        annotations       = np.load(ann_path, allow_pickle=True)
        annotation_colors = np.load(col_path, allow_pickle=True)
        return annotations, annotation_colors, xs, ys

    print(f"  [make_annotations] Generating annotations for M{mouse}...")
    annotations = np.zeros((len(ys), len(xs)), dtype=object)
    for yi, y in enumerate(ys):
        for xi, x in enumerate(xs):
            coord_SC, coord_CCF  = brain_coord_from_xy(x, y, adjusted_probe_locs_SC, shank_id=0)
            z_CCF, y_CCF, x_CCF = np.round(coord_CCF / 10).astype(int)
            annotation_index     = annotations_set[z_CCF, y_CCF, x_CCF]
            if len(structure_set[structure_set['id'] == annotation_index]) == 1:
                annotation = structure_set[structure_set['id'] == annotation_index]['acronym'].iloc[0]
            else:
                annotation = 'root'
            annotations[yi, xi] = annotation

    annotation_colors = get_annotation_colors_2D(annotations)
    np.save(ann_path, annotations)
    np.save(col_path, annotation_colors)
    print(f"  [make_annotations] Saved to {path}")
    return annotations, annotation_colors, xs, ys


In [3]:
# --- Step 1: regenerate annotation files for every unique mouse ---
print("=== Regenerating annotation files (force_remake=True) ===")
for mouse in mouse_days:
    try:
        make_annotations(mouse, path=data_path, force_remake=True)
    except Exception as e:
        print(f"  WARNING: could not make annotations for M{mouse}: {e}")


=== Regenerating annotation files (force_remake=True) ===
  [make_annotations] Removed cache: /Users/harryclark/Documents/spatial-manifolds/data/M20_annotations.npy
  [make_annotations] Removed cache: /Users/harryclark/Documents/spatial-manifolds/data/M20_annotation_colors.npy
  [make_annotations] Generating annotations for M20...
  [make_annotations] Saved to /Users/harryclark/Documents/spatial-manifolds/data
  [make_annotations] Removed cache: /Users/harryclark/Documents/spatial-manifolds/data/M21_annotations.npy
  [make_annotations] Removed cache: /Users/harryclark/Documents/spatial-manifolds/data/M21_annotation_colors.npy
  [make_annotations] Generating annotations for M21...
  [make_annotations] Saved to /Users/harryclark/Documents/spatial-manifolds/data
  [make_annotations] Generating annotations for M22...
  [make_annotations] Saved to /Users/harryclark/Documents/spatial-manifolds/data
  [make_annotations] Removed cache: /Users/harryclark/Documents/spatial-manifolds/data/M25_ann

In [ ]:
def plot_cells_on_annotated_probe(mouse, day, gcs, ngs, all_cells,
                                  path=data_path, out_dir=fig_out):
    """
    Plot all cells on the annotated probe map.
      Grey  : cells that are neither GC nor NGS
      Blue  : NGS  (#3171ae)
      Red   : GC   (#c04744)
    """
    gc_ids      = set(gcs.cluster_id.values)
    ngs_ids     = set(ngs.cluster_id.values)
    other_cells = all_cells[~all_cells.cluster_id.isin(gc_ids | ngs_ids)]

    print(f'  M{mouse} D{day}: {len(gcs)} GC  {len(ngs)} NGS  '
          f'{len(other_cells)} other  ({len(all_cells)} total)')

    annotations, annotation_colors, xs, ys = make_annotations(
        mouse, path=path, force_remake=False
    )
    unique_colors = np.unique(annotation_colors)

    fig, ax = plt.subplots(figsize=(6, 10))

    for color_ in unique_colors:
        border_points = extract_border(annotation_colors, color_, only_border=False)
        ax.scatter(xs[border_points[:, 1]], ys[border_points[:, 0]],
                   color=color_, s=1, rasterized=True)

    plot_NP2_probe(ax, sorting_analyzer_path='/Users/harryclark/Downloads/kilosort4_sa',
                   probe_alpha=0.5, contacts_alpha=0.5,
                   probe_color='grey', probe_edgecolor='black')

    if len(other_cells) > 0:
        ax.scatter(other_cells.probe_x.values, other_cells.probe_y.values,
                   marker='o', color='grey', alpha=0.3, s=10, label='Other')
    if len(ngs) > 0:
        ax.scatter(ngs.probe_x.values, ngs.probe_y.values,
                   marker='o', color='#3171ae', alpha=0.7, s=30, label='NGS')
    if len(gcs) > 0:
        ax.scatter(gcs.probe_x.values, gcs.probe_y.values,
                   marker='o', color='#c04744', alpha=0.7, s=30, label='GC')

    ax.legend(loc='upper right', fontsize=8, framealpha=0.7)
    ax.set_ylim([0, 3000])
    ax.set_xlim([-100, 900])
    ax.set_xlabel('Probe X (µm)')
    ax.set_ylabel('Probe Y (µm)')
    ax.set_title(f'M{mouse}  D{day}')

    out_path = f'{out_dir}/M{mouse}D{day}_probe_map.pdf'
    plt.savefig(out_path, bbox_inches='tight')
    plt.close()
    print(f'  Saved → {out_path}')


# --- Step 2: classify cells and plot probe maps for every mouse-day ---
print("=== Classifying cells and plotting probe maps ===")
for mouse, days in mouse_days.items():
    for day in days:
        try:
            gcs, ngs, all_cells = classify_cells_both_sessions(
                mouse=mouse, day=day, percentile_threshold=95, verbose=False
            )
            plot_cells_on_annotated_probe(mouse, day, gcs, ngs, all_cells)
        except Exception as e:
            print(f"  ERROR M{mouse} D{day}: {e}")


=== Classifying cells and plotting probe maps ===
  M20 D14: 4 GC  109 NGS  86 other  (199 total)
hi
  Saved → /Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_1_registration/M20D14_probe_map.pdf
  M20 D15: 3 GC  50 NGS  12 other  (65 total)
hi
  Saved → /Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_1_registration/M20D15_probe_map.pdf
  M20 D16: 11 GC  165 NGS  33 other  (209 total)
hi
  Saved → /Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_1_registration/M20D16_probe_map.pdf
  M20 D17: 3 GC  70 NGS  34 other  (107 total)
hi
  Saved → /Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_1_registration/M20D17_probe_map.pdf
  M20 D18: 4 GC  105 NGS  24 other  (133 total)
hi
  Saved → /Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_1_registration/M20D18_probe_map.pdf
  M20 D19: 1 GC  91 NGS  16 other  (108 total)
hi
  Saved → /Users/harryclark/Documents/spatial-manifolds/scripts/figures/

: 